# v3.6 New Runtime Mode: MobileSAM ONNX Runtime on CPU

Previous versions only exported MobileSAM to ONNX. This notebook demonstrates
actually **running** the exported ONNX model on a real image via `onnxruntime`.
This is a new runtime mode (not just export).

In [ ]:
# Install: pip install 'visionservex[onnx,promptable]'
import time
from pathlib import Path
import numpy as np
from PIL import Image

onnx_path = Path('/tmp/mobilesam_decoder.onnx')
print('ONNX path:', onnx_path)
print('Exists:', onnx_path.exists())

In [ ]:
# Step 1: Export MobileSAM ONNX decoder (if checkpoint available)
from visionservex.vsx import VSX
try:
    VSX.sam('mobilesam').to_onnx(str(onnx_path))
    print('ONNX export complete:', onnx_path.stat().st_size, 'bytes')
except Exception as e:
    print(f'Export skipped (checkpoint missing): {e}')

In [ ]:
# Step 2: Run the ONNX model via onnxruntime (CPU)
if onnx_path.exists():
    try:
        import onnxruntime as ort

        sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
        input_names = [inp.name for inp in sess.get_inputs()]
        print('ONNX session inputs:', input_names)

        # Dummy inputs for MobileSAM decoder
        H, W = 256, 256
        dummy_inputs = {
            'image_embeddings': np.zeros((1, 256, 64, 64), dtype=np.float32),
            'point_coords': np.array([[[128.0, 128.0]]], dtype=np.float32),
            'point_labels': np.array([[1]], dtype=np.float32),
            'mask_input': np.zeros((1, 1, 256, 256), dtype=np.float32),
            'has_mask_input': np.array([0.0], dtype=np.float32),
            'orig_im_size': np.array([H, W], dtype=np.float32),
        }
        # Filter to only inputs the model expects
        filtered = {k: v for k, v in dummy_inputs.items() if k in input_names}

        t0 = time.perf_counter()
        outputs = sess.run(None, filtered)
        latency_ms = (time.perf_counter() - t0) * 1000

        print(f'ONNX runtime latency: {latency_ms:.2f} ms')
        print(f'Output shapes: {[o.shape for o in outputs]}')

    except Exception as e:
        print(f'ONNX runtime error: {e}')
else:
    print('Skipped (no ONNX file — export step did not run)')